## Unindo TABELAS LOJAS e VENDAS

### IMPORTs

In [1]:
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import boto3
import io
import os

### Pegando arquivos da AWS [ S3 ]

In [2]:
s3_client = boto3.client('s3')

# nome do bucket
bucket_name = 'lojas-vendedores-carro-etl-projeto' 

# Caminhos (Keys) dos arquivos no S3
key_vendas = 'prata/vendas/vendas_carros_prata.parquet'
key_lojas = 'prata/lojas/lojas_vendedores_prata.parquet'

### Função para ler do S3 para a memória

In [3]:
def ler_parquet_s3(bucket, key):
    print(f"Baixando s3://{bucket}/{key} ...")
    # Pega o objeto do S3
    response = s3_client.get_object(Bucket=bucket, Key=key)
    
    # Lê o conteúdo binário (Body) e transforma em um buffer de bytes
    parquet_bytes = io.BytesIO(response['Body'].read())
    
    # O Pandas lê esse buffer diretamente
    df = pd.read_parquet(parquet_bytes)
    return df

# ==========================================
# 3. EXTRAÇÃO (Lendo os dados Prata)
# ==========================================
df_vendas = ler_parquet_s3(bucket_name, key_vendas)
df_lojas = ler_parquet_s3(bucket_name, key_lojas)

print("\nArquivos carregados com sucesso na memória!\n")

Baixando s3://lojas-vendedores-carro-etl-projeto/prata/vendas/vendas_carros_prata.parquet ...
Baixando s3://lojas-vendedores-carro-etl-projeto/prata/lojas/lojas_vendedores_prata.parquet ...

Arquivos carregados com sucesso na memória!



### Unindo PARQUET em um DataFrame

In [4]:
print("Unindo as bases Prata para criar a tabela Ouro...")

df_ouro = pd.merge(
    df_vendas, 
    df_lojas, 
    on=['id_loja', 'id_vendedor'], 
    how='left'
)

# Ordenando cronologicamente pela data da venda e resetando o índice
df_ouro = df_ouro.sort_values(by='data_venda').reset_index(drop=True)

display(df_ouro)

Unindo as bases Prata para criar a tabela Ouro...


,id_venda,id_loja,modelo_carro,marca,valor,data_venda,id_vendedor,categoria_ticket,nome_loja,endereco,nome_vendedor
0,15,104,Lancer Evolution,Mitsubishi,174494.87,2025-01-01,8,Intermediário,Euro Sports BH,"Rua da Bahia, 300",Julia Lima
1,133,104,370Z,Nissan,401006.25,2025-01-02,7,Premium,Euro Sports BH,"Rua da Bahia, 300",Andre Costa
2,23,101,NSX,Honda,298457.26,2025-01-03,1,Intermediário,Premium Imports SP,"Av. Paulista, 1000",Kevin Soffa
3,98,103,GR Yaris,Toyota,94565.29,2025-01-04,6,Entrada,Elite Drift RJ,"Av. Atlântica, 200",Beatriz Santos
4,111,102,S2000,Honda,123671.85,2025-01-08,3,Entrada,Japan Cars Sul,"Rua Curitiba, 50",Mateus Souza
...,...,...,...,...,...,...,...,...,...,...,...
195,156,101,M3,BMW,197779.27,2026-02-02,2,Intermediário,Premium Imports SP,"Av. Paulista, 1000",Lucas Silva
196,172,102,Eclipse,Mitsubishi,430746.58,2026-10-01,3,Premium,Japan Cars Sul,"Rua Curitiba, 50",Mateus Souza
197,70,103,GT-R,Nissan,193397.66,2026-10-01,6,Intermediário,Elite Drift RJ,"Av. Atlântica, 200",Beatriz Santos
198,109,101,S2000,Honda,274372.17,2026-12-01,2,Intermediário,Premium Imports SP,"Av. Paulista, 1000",Lucas Silva


### Criando arquivo PARQUET

In [6]:
# carrega .env
load_dotenv(find_dotenv())

# Puxa o caminho da pasta configurado no .env
diretorio_base = os.getenv('CAMINHO_DIRETORIO_PARQUET')

# Cria o caminho completo (Pasta + Nome do Arquivo) para a camada ouro
caminho_ouro_local = os.path.join(diretorio_base, "vendas_consolidadas_ouro.parquet")

# Convertendo o DataFrame para Parquet e salvando no disco
df_ouro.to_parquet(caminho_ouro_local, engine='pyarrow', index=False)

print("✅ Arquivo 'vendas_consolidadas_ouro.parquet' gerado com sucesso")

✅ Arquivo 'vendas_consolidadas_ouro.parquet' gerado com sucesso
